**PREPROCESSING**

In [6]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder

# ============================================
# LOAD DATASET
# ============================================

DATASET_PATH = "/content/unified_multimodal_ecommerce_products_dataset.csv.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset Loaded Successfully")
print(df.head())

# ============================================
# SELECT REQUIRED COLUMNS
# ============================================

required_columns = [
    'product_name',
    'description',
    'product_category_tree',
    'retail_price',
    'discounted_price',
    'brand'
]

df = df[required_columns]

print("\nSelected Required Columns")
print(df.columns)

# ============================================
# HANDLE MISSING VALUES
# ============================================

df['product_name'] = df['product_name'].fillna("Unknown Product")
df['description'] = df['description'].fillna("No Description")
df['product_category_tree'] = df['product_category_tree'].fillna("Unknown Category")
df['brand'] = df['brand'].fillna("Unknown Brand")

df['retail_price'] = df['retail_price'].fillna(0)
df['discounted_price'] = df['discounted_price'].fillna(0)

print("\nMissing Values Handled")

# ============================================
# REMOVE DUPLICATES
# ============================================

initial_rows = len(df)

df.drop_duplicates(inplace=True)

final_rows = len(df)

print(f"\nDuplicates Removed: {initial_rows - final_rows}")

# ============================================
# REMOVE HTML TAGS
# ============================================

def remove_html(text):
    return BeautifulSoup(str(text), "html.parser").get_text()

df['description'] = df['description'].apply(remove_html)

print("\nHTML Tags Removed")

# ============================================
# TEXT CLEANING
# ============================================

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = str(text)

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\\S+', '', text)

    # remove special chars
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)

    # remove numbers
    text = re.sub(r'\\d+', '', text)

    # remove extra spaces
    text = text.strip()

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

# ============================================
# APPLY CLEANING
# ============================================

print("\nCleaning Product Names...")
df['clean_product_name'] = df['product_name'].apply(clean_text)

print("Cleaning Descriptions...")
df['clean_description'] = df['description'].apply(clean_text)

print("\nText Cleaning Completed")

# ============================================
# CATEGORY EXTRACTION
# ============================================

def extract_main_category(category):

    category = str(category)

    # remove brackets
    category = category.replace("[", "")
    category = category.replace("]", "")
    category = category.replace('"', "")

    # split by >>
    main_category = category.split(">>")[0]

    return main_category.strip()

df['main_category'] = df['product_category_tree'].apply(extract_main_category)

print("\nCategory Extraction Completed")

# ============================================
# LABEL ENCODING
# ============================================

label_encoder = LabelEncoder()

df['category_encoded'] = label_encoder.fit_transform(df['main_category'])

print("\nLabel Encoding Completed")

# ============================================
# REMOVE EMPTY ROWS
# ============================================

df = df[df['clean_product_name'].str.strip() != ""]
df = df[df['clean_description'].str.strip() != ""]

# ============================================
# RESET INDEX
# ============================================

df.reset_index(drop=True, inplace=True)

# ============================================
# DATASET INFORMATION
# ============================================

print("\n========== FINAL DATASET ==========")

print(df.head())

print("\nDataset Shape:")
print(df.shape)

# ============================================
# SAVE PROCESSED DATASET
# ============================================

OUTPUT_PATH = "processed_ecommerce_products.csv"

df.to_csv(OUTPUT_PATH, index=False)

print(f"\nProcessed Dataset Saved Successfully")
print(f"Saved File: {OUTPUT_PATH}")

# ============================================
# SAVE CATEGORY MAPPING
# ============================================

category_mapping = pd.DataFrame({
    'Category': label_encoder.classes_,
    'Encoded_Value': range(len(label_encoder.classes_))
})

category_mapping.to_csv("category_mapping.csv", index=False)

print("\nCategory Mapping Saved")

# ============================================
# DISPLAY CATEGORY DISTRIBUTION
# ============================================

print("\n========== TOP CATEGORIES ==========")

print(df['main_category'].value_counts().head(10))

# ============================================
# SAMPLE OUTPUT
# ============================================

print("\n========== SAMPLE CLEANED DATA ==========")

sample = df[[
    'product_name',
    'clean_product_name',
    'main_category'
]].head(5)

print(sample)

# ============================================
# COMPLETED
# ============================================

print("\n===================================")
print("DATA PREPROCESSING COMPLETED")
print("===================================")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset Loaded Successfully
                            product_name  \
0    Alisha Solid Women's Cycling Shorts   
1    FabHomeDecor Fabric Double Sofa Bed   
2                             AW Bellies   
3    Alisha Solid Women's Cycling Shorts   
4  Sicons All Purpose Arnica Dog Shampoo   

                                         description  \
0  Key Features of Alisha Solid Women's Cycling S...   
1  FabHomeDecor Fabric Double Sofa Bed (Finish Co...   
2  Key Features of AW Bellies Sandals Wedges Heel...   
3  Key Features of Alisha Solid Women's Cycling S...   
4  Specifications of Sicons All Purpose Arnica Do...   

                               product_category_tree  retail_price  \
0  ["Clothing >> Women's Clothing >> Lingerie, Sl...         999.0   
1  ["Furniture >> Living Room Furniture >> Sofa B...       32157.0   
2  ["Footwear >> Women's Footwear >> Ballerinas >...         999.0   
3  ["Clothing >> Women's Clothing >> Lingerie, Sl...         699.0   
4  ["Pet Supplies >>

### Linear Regression for Discounted Price Prediction
Linear Regression was used to predict the `discounted_price` which is a continuous numerical value. The `clean_product_name` and `clean_description` were transformed into numerical features using TF-IDF. The model was trained to find a linear relationship between these text features and the discounted price. The evaluation metrics (MSE, RMSE, R2 Score) indicate how well the model predicts the price. A negative R2 score suggests that the model does not fit the data well for this prediction task.

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

try:
    df
except NameError:
    df = pd.read_csv('/content/unified_multimodal_ecommerce_products_dataset.csv.csv')

print("Libraries imported and dataset loaded.")

Libraries imported and dataset loaded.


In [8]:

df['combined_text_features'] = df['clean_product_name'] + ' ' + df['clean_description']

tfidf_vectorizer = TfidfVectorizer(max_features=5000)

X = tfidf_vectorizer.fit_transform(df['combined_text_features'])

y = df['discounted_price']

print("Combined text features created and vectorized.")
print(f"Shape of feature matrix X: {X.shape}")
print(f"Shape of target vector y: {y.shape}")

Combined text features created and vectorized.
Shape of feature matrix X: (9510, 5000)
Shape of target vector y: (9510,)


In [9]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Data split into training and testing sets.
X_train shape: (7608, 5000)
X_test shape: (1902, 5000)


In [10]:

linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train, y_train)

print("Linear Regression model trained.")

Linear Regression model trained.


In [11]:

y_pred = linear_reg_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2) Score: {r2:.2f}")


Model Evaluation:
Mean Squared Error (MSE): 464593228.35
Root Mean Squared Error (RMSE): 21554.42
R-squared (R2) Score: -18.93


In [12]:
linear_reg_model.score(X_train,y_train)

0.9084166203212046

### Logistic Regression
Logistic Regression was employed for multi-class classification to predict the `category_encoded` (product categories). The model classifies products into one of many categories based on their `clean_product_name` and `clean_description` (transformed by TF-IDF). The accuracy and classification report (precision, recall, f1-score) indicate the model's performance in correctly categorizing products. While overall accuracy is high, some rare categories have poor performance (zero precision/recall), which is common in imbalanced datasets.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("\n--- Starting Logistic Regression ---")

y_classification = df['category_encoded']

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(X, y_classification, test_size=0.2, random_state=42)

print("Data split for Logistic Regression.")
print(f"X_train_cls shape: {X_train_cls.shape}")
print(f"y_train_cls shape: {y_train_cls.shape}")

logistic_reg_model = LogisticRegression(solver='liblinear', multi_class='auto', max_iter=1000, random_state=42)
logistic_reg_model.fit(X_train_cls, y_train_cls)

print("Logistic Regression model trained.")

y_pred_cls = logistic_reg_model.predict(X_test_cls)

accuracy = accuracy_score(y_test_cls, y_pred_cls)
report = classification_report(y_test_cls, y_pred_cls, zero_division=0)

print("\nLogistic Regression Model Evaluation:")
print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:\n", report)



--- Starting Logistic Regression ---
Data split for Logistic Regression.
X_train_cls shape: (7608, 5000)
y_train_cls shape: (7608,)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression model trained.

Logistic Regression Model Evaluation:
Accuracy: 0.93
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
          13       0.96      0.99      0.98       168
          14       0.82      0.50      0.62        18
          15       0.73      0.79      0.76        14
          16       1.00      0.98      0.99       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       0.00      0.00      0.00         5
          25       0.00      0.00      0.00         1
          27       0.91      1.00      0.95       492
          28       0.00      0.00      0.00         2
          29       0.00      0.00      0.00         1
          31       0.90      0.98      0.94        63
          32       0.00      0.00      0.00         1
          34       0.0

### Decision Tree Classifier Summary
The Decision Tree Classifier was also used for multi-class classification to predict the `category_encoded`. It works by splitting the data based on feature values to create a tree-like structure of decisions. The model learns a set of if-then-else decision rules. Its performance is evaluated by accuracy and a classification report, similar to Logistic Regression. Decision Trees are interpretable but can be prone to overfitting, which ensemble methods like Random Forest aim to address.

In [14]:
from sklearn.tree import DecisionTreeClassifier

print("\n--- Starting Decision Tree Classifier ---")

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_cls, y_train_cls)

print("Decision Tree Classifier model trained.")

y_pred_dt = decision_tree_model.predict(X_test_cls)

accuracy_dt = accuracy_score(y_test_cls, y_pred_dt)
report_dt = classification_report(y_test_cls, y_pred_dt, zero_division=0)

print("\nDecision Tree Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_dt:.2f}")
print("Classification Report:\n", report_dt)



--- Starting Decision Tree Classifier ---
Decision Tree Classifier model trained.

Decision Tree Classifier Model Evaluation:
Accuracy: 0.91
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         1
          12       0.00      0.00      0.00         0
          13       0.96      0.97      0.97       168
          14       0.57      0.67      0.62        18
          15       0.71      0.71      0.71        14
          16       0.94      0.96      0.95       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       0.50      0.20      0.29         5
          25       0.00      0.00      0.00         1
          27       0.97

### Random Forest Classifier Summary
Random Forest, an ensemble method, was applied for multi-class classification to predict the `category_encoded`. It builds multiple decision trees during training and merges their predictions to produce a more accurate and stable classification. This approach helps in reducing overfitting compared to a single decision tree. The high accuracy and improved F1-scores for several categories (compared to Decision Tree) demonstrate its effectiveness in categorizing products based on their textual features.

In [15]:
from sklearn.ensemble import RandomForestClassifier

print("\n--- Starting Random Forest Classifier ---")

random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
random_forest_model.fit(X_train_cls, y_train_cls)

print("Random Forest Classifier model trained.")

y_pred_rf = random_forest_model.predict(X_test_cls)

accuracy_rf = accuracy_score(y_test_cls, y_pred_rf)
report_rf = classification_report(y_test_cls, y_pred_rf, zero_division=0)

print("\nRandom Forest Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_rf:.2f}")
print("Classification Report:\n", report_rf)



--- Starting Random Forest Classifier ---
Random Forest Classifier model trained.

Random Forest Classifier Model Evaluation:
Accuracy: 0.93
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
          13       0.99      1.00      0.99       168
          14       1.00      0.17      0.29        18
          15       0.90      0.64      0.75        14
          16       0.96      0.96      0.96       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       1.00      0.40      0.57         5
          25       0.00      0.00      0.00         1
          27       0.93      0.99      0.96       492
          28       0.00      0.00      0.00         2
          29       0.00      0.00      0.00         1
          31       0.95      0.84      0.89        63
          32       1.00

### K-Nearest Neighbors (KNN) Classifier Summary
K-Nearest Neighbors (KNN) was used for multi-class classification to predict the `category_encoded`. As a non-parametric algorithm, KNN classifies a data point based on the majority class among its `k` nearest neighbors in the feature space. In this context, it categorizes products by finding similar products (based on TF-IDF features of product name and description) and assigning the most common category among them. Its performance is indicated by the accuracy and classification report, showing its ability to group similar product descriptions into the correct categories.

In [16]:
from sklearn.neighbors import KNeighborsClassifier

print("\n--- Starting K-Nearest Neighbors Classifier ---")

knn_model = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_model.fit(X_train_cls, y_train_cls)

print("K-Nearest Neighbors Classifier model trained.")

y_pred_knn = knn_model.predict(X_test_cls)

accuracy_knn = accuracy_score(y_test_cls, y_pred_knn)
report_knn = classification_report(y_test_cls, y_pred_knn, zero_division=0)

print("\nK-Nearest Neighbors Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_knn:.2f}")
print("Classification Report:\n", report_knn)



--- Starting K-Nearest Neighbors Classifier ---
K-Nearest Neighbors Classifier model trained.

K-Nearest Neighbors Classifier Model Evaluation:
Accuracy: 0.93
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         1
          13       0.94      0.97      0.96       168
          14       0.86      0.67      0.75        18
          15       0.82      1.00      0.90        14
          16       0.93      0.98      0.96       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       1.00      0.40      0.57         5
          23       0.00      0.00      0.00         0
          25       0.00      0.00      0.00         1
          26       0.00      0.00      0.00         0
          27       0.96      0.99      0.97       492
     

### Support Vector Machine (SVM) Classifier Summary
The Support Vector Machine (SVM) Classifier, specifically `LinearSVC` (a linear SVM for classification), was used for multi-class classification to predict the `category_encoded`. SVMs work by finding an optimal hyperplane that best separates the different product categories in a high-dimensional feature space. The model's high accuracy and overall strong performance in the classification report (precision, recall, f1-score) suggest that it effectively distinguished between various product categories based on their clean textual features.

In [17]:
from sklearn.svm import SVC


from sklearn.svm import LinearSVC


svm_model = LinearSVC(random_state=42, dual=False)
svm_model.fit(X_train_cls, y_train_cls)

print("Support Vector Machine Classifier model trained.")

y_pred_svm = svm_model.predict(X_test_cls)

accuracy_svm = accuracy_score(y_test_cls, y_pred_svm)
report_svm = classification_report(y_test_cls, y_pred_svm, zero_division=0)

print("\nSupport Vector Machine Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_svm:.2f}")
print("Classification Report:\n", report_svm)

Support Vector Machine Classifier model trained.

Support Vector Machine Classifier Model Evaluation:
Accuracy: 0.95
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         0
          13       0.97      1.00      0.99       168
          14       0.85      0.94      0.89        18
          15       0.88      1.00      0.93        14
          16       0.98      0.99      0.99       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       1.00      0.80      0.89         5
          25       0.00      0.00      0.00         1
          27       0.97      0.98      0.97       492
          28       0.33      0.50      0.40         2
          29       0.00      0.00      0.00         1
          31       0.97      0.98      0.98     

### Naive Bayes Classifier Summary
Multinomial Naive Bayes was utilized for multi-class classification to predict the `category_encoded`. This classifier is particularly well-suited for text classification tasks because it works with feature counts (like word frequencies from TF-IDF). Despite its 'naive' assumption of feature independence, it often performs efficiently and effectively. The model's accuracy and classification report show its performance in categorizing products, highlighting its ability to leverage word patterns in product names and descriptions for classification.

In [18]:
from sklearn.naive_bayes import MultinomialNB

mnb_model = MultinomialNB()
mnb_model.fit(X_train_cls, y_train_cls)

print("Naive Bayes Classifier model trained.")

y_pred_mnb = mnb_model.predict(X_test_cls)

accuracy_mnb = accuracy_score(y_test_cls, y_pred_mnb)
report_mnb = classification_report(y_test_cls, y_pred_mnb, zero_division=0)

print("\nNaive Bayes Classifier Model Evaluation:")
print(f"Accuracy: {accuracy_mnb:.2f}")
print("Classification Report:\n", report_mnb)

Naive Bayes Classifier model trained.

Naive Bayes Classifier Model Evaluation:
Accuracy: 0.89
Classification Report:
               precision    recall  f1-score   support

           1       0.00      0.00      0.00         1
           6       0.00      0.00      0.00         1
          13       0.90      1.00      0.95       168
          14       0.00      0.00      0.00        18
          15       1.00      0.21      0.35        14
          16       0.98      0.90      0.94       100
          18       0.00      0.00      0.00         1
          20       0.00      0.00      0.00         1
          22       0.00      0.00      0.00         5
          25       0.00      0.00      0.00         1
          27       0.86      1.00      0.93       492
          28       0.00      0.00      0.00         2
          29       0.00      0.00      0.00         1
          31       0.93      0.83      0.87        63
          32       0.00      0.00      0.00         1
          34    

In [20]:
import pickle

results = {
    "Logistic Regression": accuracy,
    "Decision Tree Classifier": accuracy_dt,
    "Random Forest Classifier": accuracy_rf,
    "K-Nearest Neighbors": accuracy_knn,
    "Support Vector Machine": accuracy_svm,
    "Naive Bayes": accuracy_mnb
}

print("\n--- Model Comparison (Accuracy) ---")
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")

best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]

print(f"\nThe best performing classification model is {best_model_name} with an accuracy of {best_accuracy:.4f}.")

if best_model_name == "Logistic Regression":
    best_model = logistic_reg_model
elif best_model_name == "Decision Tree Classifier":
    best_model = decision_tree_model
elif best_model_name == "Random Forest Classifier":
    best_model = random_forest_model
elif best_model_name == "K-Nearest Neighbors":
    best_model = knn_model
elif best_model_name == "Support Vector Machine":
    best_model = svm_model
elif best_model_name == "Naive Bayes":
    best_model = mnb_model
else:
    best_model = None

if best_model:
    pickle_filename = f"{best_model_name.replace(' ', '_').lower()}_model.pkl"
    with open(pickle_filename, 'wb') as file:
        pickle.dump(best_model, file)
    print(f"\nBest model '{best_model_name}' saved as '{pickle_filename}'")
else:
    print("No best model found to save.")


--- Model Comparison (Accuracy) ---
Logistic Regression: 0.9295
Decision Tree Classifier: 0.9111
Random Forest Classifier: 0.9269
K-Nearest Neighbors: 0.9343
Support Vector Machine: 0.9532
Naive Bayes: 0.8864

The best performing classification model is Support Vector Machine with an accuracy of 0.9532.

Best model 'Support Vector Machine' saved as 'support_vector_machine_model.pkl'
